In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install earthengine-api geemap rasterio scikit-image

In [ ]:
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
ee.Authenticate()
ee.Initialize(project='YOUR_EE_PROJECT_ID')

In [ ]:
california_roi = ee.Geometry.Rectangle([

    -124.5,
    36.5,

    -119.0,
    41.5
])

In [ ]:
Map = geemap.Map()

Map.centerObject(california_roi, 6)

Map.addLayer(california_roi, {}, "California ROI")

Map

In [ ]:
dem = ee.Image("USGS/SRTMGL1_003").clip(california_roi)

In [ ]:
terrain = ee.Terrain.products(dem)

slope = terrain.select("slope")

In [ ]:
Map = geemap.Map()

Map.centerObject(california_roi, 6)

Map.addLayer(
    dem,
    {
        "min": 0,
        "max": 4000,
        "palette": ["blue", "green", "yellow", "red"]
    },
    "California DEM"
)

Map

In [ ]:
Map = geemap.Map()

Map.centerObject(california_roi, 6)

Map.addLayer(
    slope,
    {
        "min": 0,
        "max": 60,
        "palette": ["white", "yellow", "orange", "red"]
    },
    "California Slope"
)

Map

In [ ]:
EXPORT_PATH = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/california/raw/dem"

os.makedirs(EXPORT_PATH, exist_ok=True)

print("California DEM export directory created.")

In [ ]:
geemap.ee_export_image(

    slope,

    filename=f"{EXPORT_PATH}/california_slope.tif",

    scale=1000,

    region=california_roi,

    file_per_band=False
)

In [ ]:
import rasterio

In [ ]:
slope_path = f"{EXPORT_PATH}/california_slope.tif"

src = rasterio.open(slope_path)

slope_array = src.read(1)

print(slope_array.shape)

In [ ]:
plt.figure(figsize=(8,6))

plt.imshow(slope_array, cmap='terrain')

plt.colorbar()

plt.title("California Raw Slope")

plt.show()

In [ ]:
slope_array = np.nan_to_num(slope_array)

In [ ]:
def normalize(x):

    return (x - x.min()) / (x.max() - x.min())

In [ ]:
slope_norm = normalize(slope_array)

In [ ]:
print("Min:", slope_norm.min())

print("Max:", slope_norm.max())

In [ ]:
plt.figure(figsize=(8,6))

plt.imshow(slope_norm, cmap='terrain')

plt.colorbar()

plt.title("Normalized California Slope")

plt.show()

In [ ]:
from skimage.transform import resize

In [ ]:
GRID_32 = (32,32)

GRID_64 = (64,64)

In [ ]:
slope_32 = resize(

    slope_norm,
    GRID_32,
    anti_aliasing=True
)

slope_64 = resize(

    slope_norm,
    GRID_64,
    anti_aliasing=True
)

In [ ]:
print(slope_32.shape)

print(slope_64.shape)

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(slope_32, cmap='terrain')

plt.colorbar()

plt.title("California 32x32 Terrain Grid")

plt.show()

In [ ]:
BASE = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/california/grids"

os.makedirs(f"{BASE}/32x32", exist_ok=True)

os.makedirs(f"{BASE}/64x64", exist_ok=True)

In [ ]:
np.save(

    f"{BASE}/32x32/terrain.npy",

    slope_32
)

print("California 32x32 terrain tensor saved.")

In [ ]:
np.save(

    f"{BASE}/64x64/terrain.npy",

    slope_64
)

print("California 64x64 terrain tensor saved.")

In [ ]:
grid_path = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/california/grids/32x32"

print(os.listdir(grid_path))

In [ ]:
terrain = np.load(

    f"{grid_path}/terrain.npy"
)

print(terrain.shape)

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(terrain, cmap='terrain')

plt.colorbar()

plt.title("Final California Terrain Tensor")

plt.show()